In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from scipy.stats import uniform, randint
from tensorflow.keras import regularizers

In [2]:
df = pd.read_csv('merge/result/E1_AB오버샘플링.csv')

df

,기준년월,ID,Segment,입회일자_신용,수신거부여부_TM,유효카드수_체크,이용가능카드수_신용체크,이용카드수_신용체크,이용금액_R3M_신용체크,_2순위카드이용금액,...,혜택수혜금액_R3M,월중평잔,평잔_일시불_6M,인입일수_ARS_R6M,방문후경과월_앱_R6M,불만제기후경과월_R12M,컨택건수_이용유도_TM_R6M,컨택건수_이용유도_EM_R6M,잔액_신판ca최대한도소진율_r6m,변동률_RV일시불평잔
0,201810,TRAIN_315367,E,20140401,1,0,4,3,8779,4463,...,93,1202,1001,0,0,12,3,0,0.027420,0.999998
1,201810,TRAIN_211462,E,20030101,0,0,0,0,0,0,...,0,0,0,0,6,0,0,1,0.000000,0.000000
2,201811,TRAIN_248323,E,19990201,0,0,1,1,20308,0,...,0,7041,3351,7,6,12,0,5,0.230506,0.999998
3,201812,TRAIN_170447,E,20090501,0,0,1,0,0,0,...,0,0,0,0,6,0,0,1,0.017680,0.999998
4,201811,TRAIN_334511,E,20110301,0,0,1,1,696,0,...,150,420,273,7,6,12,0,1,0.049535,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2418879,201807,TRAIN_137615,E,20180601,0,0,1,1,32131,0,...,0,7572,10568,0,0,0,0,16,0.922866,0.999998
2418880,201812,TRAIN_113608,E,20040601,0,1,2,1,27008,0,...,157,4907,6941,0,6,12,0,3,0.129859,0.999998
2418881,201810,TRAIN_399526,C,20140501,1,1,3,2,92819,29718,...,524,42863,16030,0,6,0,0,5,0.210141,0.999998
2418882,201812,TRAIN_110337,D,20150301,1,1,1,0,0,0,...,0,9560,0,0,6,12,0,0,0.000000,0.999998


In [3]:
# ID, Segment 분리
X = df.drop(columns=['ID', 'Segment'])
y = df['Segment']

# Label Encoding (Segment가 문자일 경우)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 데이터 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 학습/검증 분리
X_train, X_valid, y_train, y_valid = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [4]:
# 모델 생성 함수
def create_model(hidden_units=128, dropout_rate=0.4, learning_rate=0.001):
    model = Sequential()
    
    # 첫 번째 은닉층: L2 정규화 추가
    model.add(Dense(hidden_units,
                    input_shape=(X_train.shape[1],),
                    activation='relu',
                    kernel_regularizer=regularizers.l2(0.001)))
    model.add(Dropout(dropout_rate))

    # 두 번째 은닉층: L2 정규화 추가
    model.add(Dense(hidden_units // 2,
                    activation='relu',
                    kernel_regularizer=regularizers.l2(0.001)))
    model.add(Dropout(dropout_rate))

    # 출력층 (다중 분류용)
    model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

    # 컴파일
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [5]:
# KerasClassifier로 감싸기
model = KerasClassifier(build_fn=create_model, verbose=0)

C:\Users\Lee\AppData\Local\Temp\ipykernel_20412\343470949.py:2: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasClassifier(build_fn=create_model, verbose=0)


In [6]:
# 파라미터 튜닝 범위 정의
param_dist = {
    'hidden_units': [64, 128, 256],
    'dropout_rate': uniform(0.2, 0.3),
    'learning_rate': uniform(0.0005, 0.005),
    'epochs': [10, 20],
    'batch_size': [64, 128]
}

# RandomizedSearchCV 설정
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=5,
    cv=2,
    error_score='raise',
    verbose=2,  
    random_state=42,
    n_jobs=1
)

In [7]:
# 학습
search.fit(X_train, y_train)

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END batch_size=64, dropout_rate=0.4389628960580698, epochs=10, hidden_units=256, learning_rate=0.0043984550013638464; total time= 4.8min
[CV] END batch_size=64, dropout_rate=0.4389628960580698, epochs=10, hidden_units=256, learning_rate=0.0043984550013638464; total time= 4.2min
[CV] END batch_size=64, dropout_rate=0.24680559213273096, epochs=10, hidden_units=256, learning_rate=0.0007904180608409974; total time= 4.3min
[CV] END batch_size=64, dropout_rate=0.24680559213273096, epochs=10, hidden_units=256, learning_rate=0.0007904180608409974; total time= 4.9min
[CV] END batch_size=128, dropout_rate=0.30011258334170654, epochs=20, hidden_units=256, learning_rate=0.0006029224714790123; total time= 4.7min
[CV] END batch_size=128, dropout_rate=0.30011258334170654, epochs=20, hidden_units=256, learning_rate=0.0006029224714790123; total time= 4.8min
[CV] END batch_size=128, dropout_rate=0.41659963168004743, epochs=20, hidden_units

,estimator,<keras.wrappe...002183478A440>
,param_distributions,"{'batch_size': [64, 128], 'dropout_rate': <scipy.stats....002183478A080>, 'epochs': [10, 20], 'hidden_units': [64, 128, ...], ...}"
,n_iter,5
,scoring,None
,n_jobs,1
,refit,True
,cv,2
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,'raise'


In [8]:
# 검증 성능 확인
y_pred = search.best_estimator_.predict(X_valid)
acc = accuracy_score(y_valid, y_pred)
print(f"Validation Accuracy: {acc:.4f}")
print("Best Parameters:", search.best_params_)

15119/15119 [==============================] - 10s 676us/step
Validation Accuracy: 0.8762
Best Parameters: {'batch_size': 128, 'dropout_rate': 0.41659963168004743, 'epochs': 20, 'hidden_units': 128, 'learning_rate': 0.0005038938292050716}


In [10]:
# 테스트 데이터 전처리
test = pd.read_parquet('merge/result/Segment_merge_test_ver_03.parquet')
X_test = test.drop(columns=['ID'])
X_test_scaled = scaler.transform(X_test)

In [11]:
# 예측
y_test_pred = search.best_estimator_.predict(X_test_scaled)

18750/18750 [==============================] - 14s 726us/step


In [12]:
# 저장
submission = pd.DataFrame({
    'ID': test['ID'],
    'Predicted_Segment': label_encoder.inverse_transform(y_test_pred)
})
submission.to_csv('merge/result/E2_XGb_예측.csv', index=False)